# HDFC Bank — PDF Table Discovery

Inspects HDFC_investor_PPT.pdf and HDFC_Bank_CASA.pdf using Docling.
Output: Table indices, column names, row labels — to inform hdfc_extractor.py mappings.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import OcrAutoOptions, PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

DATA_DIR = ROOT / "data"
HDFC_INV = DATA_DIR / "HDFC_investor_PPT.pdf"
HDFC_CASA = DATA_DIR / "HDFC_Bank_CASA.pdf"

pipeline_opts = PdfPipelineOptions(
    ocr_options=OcrAutoOptions(force_full_page_ocr=True),
    images_scale=2.0,
)
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts)
    }
)

In [ ]:
def load_tables(path):
    result = converter.convert(str(path))
    doc = result.document
    tables = [t.export_to_dataframe(doc=doc) for t in doc.tables]
    return tables

print("Loading HDFC Investor PDF...")
inv_tables = load_tables(HDFC_INV)
print(f"Investor: {len(inv_tables)} tables")

print("Loading HDFC CASA PDF...")
casa_tables = load_tables(HDFC_CASA)
print(f"CASA: {len(casa_tables)} tables")

In [ ]:
def inspect_tables(tables, label="PDF"):
    import pandas as pd
    for i, df in enumerate(tables):
        if df is None or df.empty:
            print(f"\n--- {label} Table {i}: empty ---")
            continue
        print(f"\n--- {label} Table {i} --- shape={df.shape}")
        print("Columns:", list(df.columns))
        print("\nFirst 10 rows (first 3 cols):")
        pd.set_option("display.max_columns", 3)
        pd.set_option("display.width", 200)
        print(df.iloc[:10, :3].to_string())
        pd.reset_option("display.max_columns")
        pd.reset_option("display.width")
        # Full text sample for label matching
        text_sample = df.astype(str).to_string()[:800]
        print(f"\nText sample (first 800 chars): {text_sample[:800]}...")

inspect_tables(inv_tables, "Investor")
inspect_tables(casa_tables, "CASA")

## KPI-to-Table Mapping (to be filled from discovery output)

Based on the output above, document which table index contains:
- Business, Deposits, Advances, CD Ratio
- Deposits breakup (Savings, Current, CASA, Term)
- Advances breakup (Retail, Agriculture, MSME, RAM, Corporate)
- NPA amounts and %
- P&L (Operating Profit, Net Profit, NII, Interest Income, etc.)
- Ratios (RoE, ROA, NIM, Cost of Deposits, etc.)